**Data Cleaning & Preprocessing**

**Import Libraries**

In [ ]:
# Import Libraries
import numpy as np
import matplotlib as plt
import seaborn as sns
import pandas as pd

**Load dataset**

In [ ]:
#Load data to Colab
from google.colab import drive
drive.mount('/content/drive')

# Import file csv to Colab
import pandas as pd
payment = pd.read_csv('/content/drive/MyDrive/Transaction Payment-Performance_Python/payment_report.csv')
product = pd.read_csv('/content/drive/MyDrive/Transaction Payment-Performance_Python/product.csv')
transactions = pd.read_csv('/content/drive/MyDrive/Transaction Payment-Performance_Python/transactions.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Display the first 5 rows of the each table**

In [ ]:
# Display the first 5 rows of the payment report table
payment_report.head()

,report_month,payment_group,product_id,source_id,volume
0,2023-01,payment,12,45,624110375
1,2023-01,payment,17,45,335715113
2,2023-01,payment,18,45,737784466
3,2023-01,payment,19,45,120963069
4,2023-01,payment,20,45,319653158


In [ ]:
# Display the first 5 rows of the product table
product.head()

,product_id,category,team_own
0,17,PXXXXXB,ASD
1,18,PXXXXXB,ASD
2,20,PXXXXXB,ASD
3,287,PXXXXXB,ASD
4,372,PXXXXXB,ASD


In [ ]:
# Display the first 5 rows of the transactions table
transactions.head()

,transaction_id,merchant_id,volume,transType,transStatus,sender_id,receiver_id,extra_info,timeStamp
0,3002692434,5,100000,24,1,10199794.0,199794.0,NaN,1682932054455
1,3002692437,305,20000,2,1,14022211.0,14022211.0,NaN,1682932054912
2,3001960110,7255,48605,22,1,NaN,10530940.0,NaN,1682932055000
3,3002680710,2270,1500000,2,1,10059206.0,59206.0,NaN,1682932055622
4,3002680713,2275,90000,2,1,10004711.0,4711.0,NaN,1682932056197


**Checked Dataset Structure**

In [ ]:
# Check the structure and data types of the payment_report dataset
payment_report.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 919 entries, 0 to 918
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   report_month   919 non-null    object
 1   payment_group  919 non-null    object
 2   product_id     919 non-null    int64 
 3   source_id      919 non-null    int64 
 4   volume         919 non-null    int64 
dtypes: int64(3), object(2)
memory usage: 36.0+ KB


In [ ]:
# Check the structure and data types of the product dataset
product.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 492 entries, 0 to 491
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   product_id  492 non-null    int64 
 1   category    492 non-null    object
 2   team_own    492 non-null    object
dtypes: int64(1), object(2)
memory usage: 11.7+ KB


In [ ]:
# Check the structure and data types of the transactions dataset
transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1324002 entries, 0 to 1324001
Data columns (total 9 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   transaction_id  1324002 non-null  int64  
 1   merchant_id     1324002 non-null  int64  
 2   volume          1324002 non-null  int64  
 3   transType       1324002 non-null  int64  
 4   transStatus     1324002 non-null  int64  
 5   sender_id       1274943 non-null  float64
 6   receiver_id     1159207 non-null  float64
 7   extra_info      6095 non-null     object 
 8   timeStamp       1324002 non-null  int64  
dtypes: float64(2), int64(6), object(1)
memory usage: 90.9+ MB


**Check for Missing Values in Transactions table**

In [ ]:
#Check for Missing Values in Transactions table
transactions.isna().sum()

,0
transaction_id,0
merchant_id,0
volume,0
transType,0
transStatus,0
sender_id,49059
receiver_id,164795
extra_info,1317907
timeStamp,0


**Check for Duplicates**

In [ ]:
# Check for Duplicates in payment_report table
payment_report.duplicated().sum()

np.int64(0)

In [ ]:
# Check for Duplicates in product table
product.duplicated().sum()

np.int64(0)

In [ ]:
# Check for Duplicates in transactions table
transactions.duplicated().sum()

np.int64(28)

**Exploratory Data Analysis (EDA)**

**Handle Missing Values in Transaction table**

**1. Explore Transaction Type Values**

In [ ]:
# Explore Transaction Type Values
transactions['transType'].unique()

array([24,  2, 22,  8, 30, 14, 12])

**2. Transaction Type Distribution**

In [ ]:
# Transaction Type Distribution
transactions['transType'].value_counts()

,count
transType,
2,760783
8,342553
22,202383
30,14586
24,3677
14,10
12,10


**3. Analyze Missing Sender and Receiver by Transaction Type**

In [ ]:
# Analyze Missing Sender and Receiver by Transaction Type
profile = (
    transactions
    .groupby('transType')
    .agg(
        total=('transaction_id', 'count'),
        missing_sender=('sender_id', lambda x: x.isna().sum()),
        missing_receiver=('receiver_id', lambda x: x.isna().sum())))

profile['pct_missing_sender'] = profile['missing_sender'] / profile['total']
profile['pct_missing_receiver'] = profile['missing_receiver'] / profile['total']

profile.sort_values('total', ascending=False)

,total,missing_sender,missing_receiver,pct_missing_sender,pct_missing_receiver
transType,,,,,
2,760783,0,162208,0.000000,0.213212
8,342553,0,0,0.000000,0.000000
22,202383,34473,0,0.170335,0.000000
30,14586,14586,2587,1.000000,0.177362
24,3677,0,0,0.000000,0.000000
12,10,0,0,0.000000,0.000000
14,10,0,0,0.000000,0.000000


### Transaction Pattern (transType = 22 & 30)

- **transType = 22**
  - `receiver_id` is always present.
  - `sender_id` is often missing.
  - This suggests a **SYSTEM → USER** transaction.

- **transType = 30**
  - `sender_id` is always missing.
  - This indicates the transaction is **initiated by the SYSTEM**.

👉 Therefore, when `sender_id` is missing and `transType ∈ {22, 30}`,  
the missing sender can be inferred as **SYSTEM**.

In [ ]:
# Handle Missing Sender for System Transactions (transType 22, 30) by Fill SYSTEM = 0
SYSTEM_ID = 0

mask = (transactions['transType'].isin([22, 30])) & transactions['sender_id'].isna()
transactions.loc[mask, 'sender_id'] = SYSTEM_ID

### Transaction Pattern (transType = 2)

- `missing_sender` = 0  
- `missing_receiver` ≈ 21%

This represents a **User → Merchant** transaction:

- **Sender:** User (always present)  
- **Receiver:** Merchant (sometimes not logged)

In [ ]:
# Handle Missing Receiver for User → Merchant Transactions (transType = 2) by Filling with merchant_id
mask = (transactions['transType'] == 2) & transactions['receiver_id'].isna()
transactions.loc[mask, 'receiver_id'] = transactions.loc[mask, 'merchant_id']

For **transType = 30**, some `receiver_id` values are missing. Since these transactions often occur close in time and typically involve the same receiver, the data was sorted by `timeStamp` and missing values were filled using nearby transactions (`ffill` and `bfill`).

In [ ]:
# Handle Missing Receiver for System Transactions (transType = 30) by Filling from Nearby Transactions (ffill & bfill)

# Filter transactions with transType = 30
df = transactions[transactions['transType'] == 30].copy()

# Sort by timestamp to maintain chronological order
df = df.sort_values('timeStamp')

# Fill missing receiver_id using forward fill and backward fill
df['receiver_id'] = df['receiver_id'].ffill().bfill()

# Update the original transactions table
transactions.update(df)

**Handle Duplicates**

In [ ]:
# Drop duplicates
transactions.drop_duplicates(subset=None, keep='first')

,transaction_id,merchant_id,volume,transType,transStatus,sender_id,receiver_id,extra_info,timeStamp,transaction_type
0,3002692434,5,100000,24,1,10199794.0,199794.0,NaN,2023-05-01 09:07:34.455,Invalid Transaction
1,3002692437,305,20000,2,1,14022211.0,14022211.0,NaN,2023-05-01 09:07:34.912,Payment Transaction
2,3001960110,7255,48605,22,1,0.0,10530940.0,NaN,2023-05-01 09:07:35.000,Invalid Transaction
3,3002680710,2270,1500000,2,1,10059206.0,59206.0,NaN,2023-05-01 09:07:35.622,Top Up Money Transaction
4,3002680713,2275,90000,2,1,10004711.0,4711.0,NaN,2023-05-01 09:07:36.197,Payment Transaction
...,...,...,...,...,...,...,...,...,...,...
1323997,3003723030,305,20000,2,1,24524311.0,305.0,NaN,2023-05-02 13:54:32.634,Payment Transaction
1323998,3003723033,2270,100000,2,1,10277242.0,277242.0,NaN,2023-05-02 13:54:32.876,Top Up Money Transaction
1323999,3003723036,2270,100000,2,1,10144599.0,144599.0,NaN,2023-05-02 13:54:32.892,Top Up Money Transaction
1324000,3003723039,5,400,22,1,10028007.0,21013762.0,NaN,2023-05-02 13:54:32.896,Invalid Transaction


**Convert Unix Timestamp to Datetime**

In [ ]:
# Convert 'timeStamp' from int64 (Unix timestamp in milliseconds) to datetime format
transactions['timeStamp'] = pd.to_datetime(transactions['timeStamp'], unit='ms')

**Merge payment_report and product DataFrames**

---



In [ ]:
#Merge payment_report with product
payment_product = payment_report.merge(product, on='product_id', how='left')
payment_product.head()

,report_month,payment_group,product_id,source_id,volume,category,team_own
0,2023-01,payment,12,45,624110375,PXXXXXT,ASD
1,2023-01,payment,17,45,335715113,PXXXXXB,ASD
2,2023-01,payment,18,45,737784466,PXXXXXB,ASD
3,2023-01,payment,19,45,120963069,PXXXXXM2,ASD
4,2023-01,payment,20,45,319653158,PXXXXXB,ASD


**Data Wrangling & Business Analysis**

**1. Top 3 product_ids with the highest volume.**

In [ ]:
# Calculate volume by product_id
volume_by_product = payment_product.groupby('product_id')['volume'].agg('sum').reset_index()

# Sort volume in descending order
volume_by_product.sort_values(by='volume', ascending=False)

# Filter the top 3 product_ids with the highest volume
top_3_productid = volume_by_product.head(3)

# Print the results
print("Top 3 product_ids with the highest volume:")
print(top_3_productid)

Top 3 product_ids with the highest volume:
   product_id      volume
0           3        6000
1          12  1934440830
2          15  4206315258


**2. Given that each product_id should be owned by only one team, are there any products that are assigned to more than one team?**

In [ ]:
# Group by product_id and count the number of unique teams
products_by_team = payment_product.groupby('product_id')['team_own'].nunique()

# Identify products owned by more than one team
abnormal_products = products_by_team[products_by_team > 1]

if abnormal_products.empty:
    print("No abnormal products found.")
else:
    print("Products owned by more than one team:")
    print(abnormal_products)

    # Retrieve detailed records for those abnormal products
    abnormal_records = payment_product[payment_product['product_id'].isin(abnormal_products.index)]

    print("\nDetailed records:")
    print(abnormal_records)

No abnormal products found.


**3. Find the team has had the lowest performance (lowest volume) since Q2.2023. Find the category that contributes the least to that team.**

In [ ]:
# Convert report_month to datetime
payment_product["report_month"] = pd.to_datetime(payment_product["report_month"])

# Filter data from Q2 2023 onwards
since_Q2_2023 = payment_product[payment_product["report_month"] >= "2023-04-01"]

# 1. Find the team with the lowest performance (lowest total volume since Q2 2023)
team_volume = since_Q2_2023.groupby("team_own")["volume"].sum().sort_values()

lowest_team_name = team_volume.index[0]
lowest_volume = team_volume.iloc[0]

print(f"The lowest performance team since Q2 2023 is {lowest_team_name} with volume = {lowest_volume}")

# 2. Find the category contributing the least to that team
lowest_team_data = since_Q2_2023[since_Q2_2023["team_own"] == lowest_team_name]

category_volume = lowest_team_data.groupby("category")["volume"].sum().sort_values()

lowest_category = category_volume.index[0]
lowest_category_volume = category_volume.iloc[0]

print(f"The lowest contributing category for team {lowest_team_name} is {lowest_category} with volume = {lowest_category_volume}")

# 3. Calculate contribution of that category within the team
contribution_pct = lowest_category_volume / lowest_team_data["volume"].sum() * 100

print(f"Contribution of {lowest_category} to team {lowest_team_name}: {contribution_pct:.2f}%")

The lowest performance team since Q2 2023 is APS with volume = 51141753
The lowest contributing category for team APS is PXXXXXE with volume = 25232438
Contribution of PXXXXXE to team APS: 49.34%


**4. Find the contribution of source_ids of refund transactions (payment_group = ‘refund’), what is the source_id with the highest contribution?**

In [ ]:
# Filter refund transactions
refund = payment_product[payment_product['payment_group'] == 'refund']

# Calculate total volume by source_id
volume_by_id = (
    refund.groupby('source_id')['volume']
    .sum()
    .reset_index()
)

# Calculate contribution percentage
volume_by_id['contribution_pct'] = (
    volume_by_id['volume'] / volume_by_id['volume'].sum() * 100
)

# Sort by contribution
volume_by_id = volume_by_id.sort_values(by='contribution_pct', ascending=False)

# Get the highest contributor
top_source = volume_by_id.iloc[0]

print(volume_by_id)

print(f"{top_source['source_id']} is the highest contributor to refund transactions with {top_source['contribution_pct']:.2f}%")


   source_id       volume  contribution_pct
1         38  36527454759         59.108225
2         39  16119059662         26.083641
0         37   9151069226         14.808134
38.0 is the highest contributor to refund transactions with 59.11%


Using transactions.csv


**5. Define type of transactions (‘transaction_type’) for each row, given:**
- transType = 2 & merchant_id = 1205: Bank Transfer Transaction
- transType = 2 & merchant_id = 2260: Withdraw Money Transaction
- transType = 2 & merchant_id = 2270: Top Up Money Transaction
- transType = 2 & others merchant_id: Payment Transaction
- transType = 8, merchant_id = 2250: Transfer Money Transaction
- transType = 8 & others merchant_id: Split Bill Transaction
- Remained cases are invalid transactions

In [ ]:
# Define a function to classify transactions based on transType and merchant_id
def classify_transaction(transType, merchant_id):
    if transType == 2:
        # Specific merchant rules for transType = 2
        if merchant_id == 1205:
            return "Bank Transfer Transaction"
        elif merchant_id == 2260:
            return "Withdraw Money Transaction"
        elif merchant_id == 2270:
            return "Top Up Money Transaction"
        else:
            # Other merchants under transType = 2 are treated as payment transactions
            return "Payment Transaction"
    elif transType == 8:
        # Specific merchant rule for transType = 8
        if merchant_id == 2250:
            return "Transfer Money Transaction"
        else:
            # Other merchants under transType = 8 correspond to split bill transactions
            return "Split Bill Transaction"
    else:
        # Transactions that do not match the defined rules
        return "Invalid Transaction"


# Apply the classification function to create the transaction_type column
# Using itertuples() improves performance compared to row-wise apply()
transactions['transaction_type'] = [
    classify_transaction(t.transType, t.merchant_id)
    for t in transactions.itertuples()
]

# Validate the classification by checking transactions with transType = 2 and merchant_id = 1205
transactions[
    (transactions['transType']==2) & (transactions['merchant_id']==1205)
][['transType','merchant_id','transaction_type']].head()

,transType,merchant_id,transaction_type
35,2,1205,Bank Transfer Transaction
89,2,1205,Bank Transfer Transaction
133,2,1205,Bank Transfer Transaction
137,2,1205,Bank Transfer Transaction
268,2,1205,Bank Transfer Transaction


**6. Of each transaction type (excluding invalid transactions): find the number of transactions, volume, senders and receivers.**

In [ ]:
# Remove invalid transactions
valid_transactions = transactions[transactions['transaction_type'] != "Invalid Trans"]

# Group by transaction type and aggregate the required metrics
transaction_summary = valid_transactions.groupby("transaction_type").agg(
    num_transactions = ("transaction_id", "count"),
    total_volume = ("volume", "sum"),
    num_senders = ("sender_id", "nunique"),
    num_receivers = ("receiver_id", "nunique")
)

# Print the final result
transaction_summary.reset_index()

,transaction_type,num_transactions,total_volume,num_senders,num_receivers
0,Bank Transfer Transaction,37879,50605806190,23156,9272
1,Invalid Transaction,220666,24659408387,2709,89716
2,Payment Transaction,398677,71851515181,139583,113784
3,Split Bill Transaction,1376,4901464,1323,572
4,Top Up Money Transaction,290502,108606478829,110409,110409
5,Transfer Money Transaction,341177,37033171492,39021,34585
6,Withdraw Money Transaction,33725,23418181420,24814,24814
